[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/sundial/sundial.ipynb)

# Sundials

This example computes the hour lines and the seasons' shadow paths of a horizontal sundial. A sundial tells the time by a shadow. Its gnomon has an edge, the style, that points at the celestial pole, the spot in the sky the earth's axis points to; the sky, and the sun with it, turns once a day about that axis, and the style's shadow turns with it, along a fixed line of the dial for each hour. The tip of the gnomon, the nodus, casts a point whose path over a day tells the season.

The sun's direction is modelled as an antivector, two turns of one direction on the celestial equator, rotors in the planes two directions span: `turn(equator, pole, declination)` towards the pole by the season's angle, then `turn(equator, west, hour)` about the pole by the hour. A shadow is where the ray from the sun past the nodus meets the dial, the plane `dial`, each point's height above it its pairing `dial & point`, and a day's rays sweep a cone about the pole, a quadric of rays, `across * (across & Point) - (across & sun).squared() * Point.dual_inverse()` with `across` the plane across the pole, through the sun about the pole, whose cut with the dial is the season's shadow path, a conic.

In the notation of spherical astronomy the sun's altitude reads as $\sin h = \sin\varphi \sin\delta + \cos\varphi \cos\delta \cos H$, at latitude $\varphi$, declination $\delta$ and hour angle $H$, and in the notation of gnomonics a horizontal dial's hour line makes the angle $\arctan(\sin\varphi \tan H)$ with the noon line.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext
from numga.algebras import VGA3D as ga
from examples.animation import save_figure, show_animation
from examples.geometry.sundial import render

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# Directions, to the sun, the pole and the compass points, and points of the dial: antivectors with
# intrinsic orientation.
Point = ga.gatype.antivector()
Plane = ga.gatype.vector()
Rotor = ga.gatype.rotor()
# A quadric of rays: the planes that read them.
Cone = ga.gatype((Plane, Point))                           # Plane <- Point
# The local frame at the dial, and the dial: the horizontal plane through the origin.
east, north, up = mv.basis().dual()                        # [] Point each
west = -east
dial = mv.z                                                # [] Plane

# The dial's latitude, the height of the nodus above the dial, and the tilt of the earth's axis, the sun's
# declination at the solstices.
latitude = np.radians(52.0)
height = 1.0
tilt = np.radians(23.44)


def hour_angle(hours: np.ndarray) -> np.ndarray:
    """The sun's hour angle at each hour of local solar time: a fifteenth of a quarter turn per hour from
    noon, westwards in the afternoon."""
    return np.radians(15.0 * (hours - 12))

## 1. The sun's two turns

The celestial pole is north raised towards the zenith by the latitude, and the celestial equator's highest point lies a quarter turn further. The sun at noon on the equinoxes stands there. Any other sun is that point turned twice: towards the pole by the declination, the season's angle north or south of the equator, then about the pole towards the west by the hour angle, a twenty-fourth of a turn per hour. At noon the sun stands a quarter turn less the latitude, plus the declination, above the horizon.

In [ ]:
def turn(start: Point, towards: Point, angle: np.ndarray) -> Rotor:
    """The turn by an angle in the plane of two directions at right angles, from the first towards the
    second: their commutator is the plane they span."""
    return (towards.commutator(start) * (-angle / 2)).exp()


def sky(latitude: float) -> tuple[Point, Point]:
    """The celestial pole, north raised towards up by the latitude, and the celestial equator's highest
    point, a quarter turn further."""
    return turn(north, up, latitude) >> north, turn(north, up, latitude + np.pi / 2) >> north


def sun(pole: Point, equator: Point, declinations: np.ndarray, hours: np.ndarray) -> Point:
    """The direction to the sun at each declination and hour angle: the equator's highest point turned
    towards the pole by the declination, then about the pole towards the west by the hour angle."""
    noon = turn(equator, pole, declinations) >> equator                       # [...] Point
    return turn(equator, west, hours) >> noon                                 # [...] Point


pole, equator = sky(latitude)
solstices_and_equinox = np.array([-tilt, 0.0, tilt])
noon_suns = sun(pole, equator, solstices_and_equinox, np.zeros(3))            # [seasons] Point
print("noon heights, winter, equinox, summer:", np.degrees(np.arcsin((dial & noon_suns).to_array())).round(2))

## 2. The hour lines

The nodus casts its shadow where the ray from the sun past the nodus meets the dial. The style runs through the nodus towards the pole, and the plane through the style and the sun at a given hour holds the sun at that hour on every day of the year: the declination turns the sun within that plane. So each hour's shadow, whatever the season, falls along one line of the dial through the foot of the style. Measured from the noon line, the hour lines crowd towards noon and spread towards morning and evening.

In [ ]:
# The hours drawn, and the declinations each hour line runs through, from solstice to solstice.
hours = np.arange(6, 19)
n_declinations = 25

In [ ]:
def shadow(nodus: Point, suns: Point) -> Point:
    """Where the shadow of the nodus falls on the dial, the horizontal plane through the origin: from the
    nodus, away from the sun, as far as the dial."""
    return nodus - suns * ((dial & nodus) / (dial & suns))                     # [...] Point


nodus = up * height                                                            # [] Point
# Where the style meets the dial.
foot = shadow(nodus, pole)                                                     # [] Point
declinations = np.linspace(-tilt, tilt, n_declinations)                    # [samples]
hour_suns = sun(pole, equator, declinations[None, :], hour_angle(hours)[:, None])   # [hours, samples] Point
hour_lines = shadow(nodus, hour_suns)                                          # [hours, samples] Point
spread = render.bearings(hour_lines - foot)                                    # [hours, samples]
print("hour line bearings from the noon line, in degrees:", np.degrees(spread[:, -1]).round(1))

## 3. The seasons as conics

On a given day the ray from the nodus to the sun keeps its angle to the pole, a quarter turn less the declination, so the day's rays sweep a cone about the pole. A cone is a quadric of rays, a map from each ray to the plane that reads it, and the ray lies on the cone where that reading of itself, `cone(ray) & ray`, is zero. The plane across the pole, `across = pole.dual_inverse()`, reads a ray's height along the pole, the sine of the declination for a unit sun, and the cone is `across * (across & Point) - (across & sun).squared() * Point.dual_inverse()`: the ray's height squared against the sun's times its own length squared, where the metric enters. The dial cuts it, and the shadow path is the cut, a conic. The form only sees the declination squared, so its other nappe holds the rays of the opposite declination: a summer day and the winter day as far from the equinox are the two branches of one hyperbola, drawn here in one colour. At the equinoxes the cone flattens into the plane of the celestial equator, and the shadow runs along a straight line.

In [ ]:
# The seasons drawn, a whole number of months from an equinox up to a solstice on either side, and the
# hours of a day sampled.
n_months = 3
day_hours = np.arange(3, 21.01, 0.05)

In [ ]:
def cone(pole: Point, sun: Point) -> Cone:
    """The cone of rays about the pole through a direction to the sun: zero on a ray exactly when it meets
    the pole at the sun's angle, or at its supplement, the other nappe."""
    # The plane across the pole reads a ray's height along it: for a unit sun, the sine of its declination.
    across = pole.dual_inverse()                                               # [...] Plane
    # A ray is on the cone when its height squared is the sun's times its own length squared, the metric.
    return across * (across & Point) - (across & sun).squared() * Point.dual_inverse()   # [...] Plane <- Point


seasons = tilt * np.sin(np.pi / 2 * np.arange(-n_months, n_months + 1) / n_months)   # [seasons]
days = sun(pole, equator, seasons[:, None], hour_angle(day_hours)[None, :])  # [seasons, samples] Point
rays = (nodus - shadow(nodus, days)).normalized()                              # [seasons, samples] Point
print("largest value of a season's cone on its day's shadow rays:", np.abs((cone(pole, days[:, 0])[:, None](rays) & rays).to_array()).max())
save_figure(render.draw_dial(nodus, foot, seasons, days, hour_suns, hours), "sundial");

## 4. A year

Through the year the declination swings between the solstices, and the cone of the day's rays opens and closes about the pole: steep in summer, its cut with the dial a tight hyperbola near the gnomon; flat at the equinoxes, a straight line; shallow in winter, the hyperbola's far branch. On the left, the day's rays from beyond the nodus down to the dial; on the right, that day's shadow path among the seasons'.

In [ ]:
# The frames of one year, the hours a frame's rays are drawn at, and the duration of a frame.
n_frames = 48
ray_hours = np.arange(4, 21, 0.25)
duration_ms = 90

In [ ]:
year_declinations = tilt * np.sin(np.linspace(0, 2 * np.pi, n_frames, endpoint=False))   # [frames]
year_rays = sun(pole, equator, year_declinations[:, None], hour_angle(ray_hours)[None, :])   # [frames, hours] Point
year_days = sun(pole, equator, year_declinations[:, None], hour_angle(day_hours)[None, :])   # [frames, samples] Point

show_animation(render.animate_year(year_declinations, year_rays, year_days, nodus, foot, seasons, days, hour_suns, hours),
               "sundial_year", duration_ms)

In [ ]:
# checks
# The noon sun stands a quarter turn less the latitude, plus the declination, above the horizon.
assert np.abs(np.arcsin((dial & noon_suns).to_array()) - (np.pi / 2 - latitude + solstices_and_equinox)).max() < 1e-10
# Every shadow ray of a season's day lies on its cone, and each hour line on the angle of gnomonics.
assert np.abs((cone(pole, days[:, 0])[:, None](rays) & rays).to_array()).max() < 1e-10
expected = np.arctan2(np.sin(latitude) * np.sin(hour_angle(hours)), np.cos(hour_angle(hours)))
assert np.abs(np.sin(spread - expected[:, None])).max() < 1e-10